# CareLoom: Product & Policy Assistant — Larkspur Electronics

Two different kinds of questions land on Larkspur's support desk every day:

- **structured facts** — "what's the warranty period on the Aria keyboard", "show me the
  notes we've saved about the webcam's known issues"
- **policy questions** — "what's our process for a warranty claim", "how long do refunds
  actually take", "what happens if a package goes missing"

The first kind is answered by looking something up. The second kind needs the actual
wording of a policy document, because "close enough" isn't good enough when a customer
is quoting a policy back at you. This notebook builds the CareLoom agent that handles
both: SQLite for structured facts, ChromaDB for semantic search over Larkspur's own
Customer Care & Policy Handbook, and a router in between that decides which one to use.

Use case:
- SQLite stores structured facts such as product metadata, saved agent notes, and query history.
- ChromaDB stores chunks from Larkspur's real policy handbook and answers semantic questions about it.
- A simple router chooses the right store based on query intent.
- LangGraph checkpointers keep the conversation thread resumable.

LangChain's SQL-agent tutorial describes the SQL agent loop of inspecting tables and schemas, deciding what is relevant, generating a query, and executing it. LangGraph's persistence docs say checkpointers persist thread-scoped state, while stores hold durable application data across threads. LangChain's knowledge-base tutorial explains the load-split-embed-store pattern for PDF search.

## Why this use case works

A support agent often needs both kinds of retrieval in the same conversation:

- structured facts such as a product's price, warranty length, or a colleague's saved note
- semantic answers from the policy handbook itself, such as what counts as a valid warranty claim or how festival-season shipping delays are handled

That makes SQLite a good home for structured data and ChromaDB a good home for the handbook's actual wording.

## Learning goals

By the end of this notebook, you should be able to:

1. Build a small, real PDF as a reproducible local artifact instead of a flaky download.
2. Chunk and embed a PDF into ChromaDB, with a offline-safe embedding fallback.
3. Query structured facts from SQLite alongside semantic search from Chroma.
4. Route a question to the right store based on simple intent rules.
5. Keep a conversation resumable with a SQLite-backed checkpointer.


## 1) Install packages


In [ ]:
%pip install -qU python-dotenv pypdf fpdf2 \
    langchain langchain-core langchain-groq \
    langchain-chroma chromadb langchain-huggingface sentence-transformers \
    langgraph langgraph-checkpoint-sqlite numpy


## 2) Load environment variables

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=careloom-product-policy-assistant
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "careloom-product-policy-assistant")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)


## 3) Prepare the reference document

The original version of this lab downloaded a public arXiv paper at this point, just to have "a real PDF" to work with. That made the notebook dependent on an internet connection and on someone else's server staying up, for a document that had nothing to do with Larkspur's business.

Here we generate the real document instead of downloading one: Larkspur's actual **Customer Care & Policy Handbook**, the reference every support agent uses. It's built as a genuine multi-page PDF with a real text layer — the same `PyPDFLoader → split → embed → store` pipeline runs on it either way — but it ships with the notebook and needs no network access. In a real team this file would already exist, exported from wherever the policy docs live (Confluence, Notion, a shared Word doc); we generate it here so everyone running this notebook gets the identical source.


In [ ]:
from pathlib import Path
from fpdf import FPDF, XPos, YPos

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

PDF_PATH = DATA_DIR / "larkspur_care_policy_handbook.pdf"

HANDBOOK_SECTIONS = [
    ("Welcome", (
        "This handbook is the shared reference for everyone on Larkspur Electronics' "
        "customer care team, whether you are a new hire in your first week or a support "
        "lead reviewing an escalation. It covers what customers are entitled to, how long "
        "each process takes, and where to draw the line between handling something "
        "yourself and looping in a supervisor. When a policy in this handbook conflicts "
        "with what a customer says a previous agent told them, this handbook is the "
        "source of truth; note the discrepancy in the ticket and flag it to your team "
        "lead so the record can be corrected.\n\n"
        "Larkspur sells keyboards, mice, headphones, monitors, webcams, and desk lighting "
        "directly to consumers across India, with a smaller volume of international "
        "orders. Most of what follows applies to both, with shipping timelines being the "
        "main exception."
    )),
    ("Returns Policy", (
        "Customers may return an unopened item within 14 days of delivery for a full "
        "refund. The item must be in its original packaging with all accessories, "
        "manuals, and included cables. A restocking fee does not apply to unopened "
        "returns.\n\n"
        "Opened items may be returned within 7 days of delivery only if the product is "
        "defective or was not as described in the listing. Not as described covers wrong "
        "color, wrong model, or missing advertised features -- it does not cover a "
        "customer simply changing their mind after opening the box. Opened, "
        "non-defective returns are not accepted; direct the customer to our exchange "
        "program instead if they want a different model.\n\n"
        "Items marked as final sale (typically clearance and open-box listings) cannot "
        "be returned under any circumstances except a manufacturing defect confirmed by "
        "our technical team. This exception always applies regardless of the final-sale "
        "label, since a defective product doesn't stop being defective because it was "
        "discounted.\n\n"
        "To start a return, the customer needs the order number and the email address "
        "used at checkout. Return shipping is free for defective items; for a "
        "change-of-mind return on an unopened item, the customer covers return shipping "
        "unless the order qualified for free returns at checkout (orders above Rs. "
        "5,000 currently qualify)."
    )),
    ("Refunds Policy", (
        "Refunds are issued within 7 business days of the returned item passing "
        "inspection at our warehouse -- not from the day the customer ships it back. "
        "Inspection typically takes 1-2 business days after the item arrives, so "
        "customers should expect roughly 8-9 business days total from drop-off to "
        "refund.\n\n"
        "Refunds are issued to the original payment method whenever possible. If the "
        "original payment method is no longer valid (an expired card, for example), the "
        "refund is issued as store credit unless the customer specifically requests a "
        "bank transfer, which takes an additional 3-5 business days to process.\n\n"
        "Refunds above Rs. 3,000 must be approved by a support lead before they are "
        "processed. This is not a reflection on the customer or the claim -- it is a "
        "standing control on any outbound payment past that threshold, and it applies "
        "the same way whether the request comes from a first-time customer or someone "
        "who has ordered from us for years. Flag the ticket for approval and let the "
        "customer know a supervisor is reviewing it; most approvals are same-day.\n\n"
        "Partial refunds are available when only part of an order is being returned, or "
        "when a customer keeps a damaged item rather than shipping it back (common for "
        "large monitors, where return shipping is expensive and slow). A partial refund "
        "in that case is typically 50-70% of the item's price, depending on the extent "
        "of the damage, and always requires supervisor approval regardless of amount."
    )),
    ("Warranty Policy", (
        "Every product Larkspur sells carries a 1-year manufacturer's warranty against "
        "defects in materials and workmanship, starting from the delivery date. This "
        "covers things like a keyboard with keys that stop registering, a mouse sensor "
        "that drifts, headphones with a dead channel, or a monitor with stuck pixels "
        "beyond the panel manufacturer's acceptable threshold.\n\n"
        "The warranty does not cover physical damage, liquid damage, damage from "
        "unauthorized repair attempts, or normal cosmetic wear such as fading keycap "
        "labels. If a customer describes dropping the product, spilling a drink on it, "
        "or opening the case themselves, that claim falls outside warranty coverage -- "
        "explain this clearly and offer a paid repair estimate instead if one is "
        "available.\n\n"
        "To file a warranty claim, the customer needs proof of purchase and a short "
        "description or photo of the issue. Our technical team reviews claims within 3 "
        "business days. Approved claims are resolved as a replacement unit, a repair, "
        "or -- if neither is available in reasonable time -- a refund at the support "
        "lead's discretion. Replacement is the default remedy; only escalate to a refund "
        "conversation if a replacement unit is genuinely unavailable within 30 days."
    )),
    ("Shipping & Delivery", (
        "Standard delivery within Chennai, Bengaluru, Mumbai, Delhi, and Pune metro "
        "limits takes 3-5 business days. Delivery outside these metro areas typically "
        "takes 5-8 business days. Expedited shipping, where available, cuts this to 1-3 "
        "business days for an additional fee shown at checkout.\n\n"
        "During major sale periods and the festival season (roughly mid-October through "
        "early November, and again in late December), delivery estimates should be "
        "extended by 2-3 additional business days across all regions due to courier "
        "network volume. Proactively mention this to customers ordering during these "
        "windows rather than waiting for them to ask why a package is running late.\n\n"
        "Every order includes tracking, sent by email at the time of dispatch. If a "
        "customer says tracking hasn't updated in more than 3 business days, check the "
        "carrier's status directly rather than only re-sending the same tracking link "
        "-- carrier-side delays are more common during festival season and courier "
        "strikes, and customers appreciate a real update over a repeated link.\n\n"
        "Orders that are lost in transit (no tracking movement for 10+ business days, "
        "confirmed with the carrier) are handled as a full replacement shipment at no "
        "cost to the customer, without needing supervisor approval -- this is a "
        "standing exception separate from the general refund-approval threshold, since "
        "the customer paid for a delivery that didn't happen."
    )),
    ("Escalation Matrix", (
        "Most tickets should be resolved by a Tier 1 agent within one interaction, using "
        "this handbook. Escalate to Tier 2 when a case needs technical judgment beyond "
        "this handbook -- for example, whether a described issue is covered under "
        "warranty when it's genuinely ambiguous.\n\n"
        "Escalate directly to a support lead (skipping Tier 2) for: any refund above Rs. "
        "3,000, any partial refund of any amount, a customer requesting a manager by "
        "name, or a customer who has contacted us three or more times about the same "
        "unresolved issue.\n\n"
        "Response-time targets: Tier 1 first response within 4 business hours; Tier 2 "
        "within 1 business day of escalation; support-lead review of a pending refund "
        "approval within the same business day it's flagged. These are targets, not "
        "guarantees, but a ticket approaching double its target time should be flagged "
        "in the daily stand-up regardless of who owns it."
    )),
]


def build_handbook_pdf(path: Path) -> Path:
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=18)
    pdf.add_page()

    pdf.set_font("Helvetica", "B", 18)
    pdf.multi_cell(0, 10, "Larkspur Electronics", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("Helvetica", "", 13)
    pdf.multi_cell(0, 8, "Customer Care & Policy Handbook", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.ln(4)

    for heading, body in HANDBOOK_SECTIONS:
        pdf.set_font("Helvetica", "B", 14)
        pdf.multi_cell(0, 8, heading, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font("Helvetica", "", 11)
        pdf.multi_cell(0, 6, body, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(4)

    pdf.output(str(path))
    return path


build_handbook_pdf(PDF_PATH)
print("Handbook PDF ready:", PDF_PATH.resolve(), f"({PDF_PATH.stat().st_size} bytes)")


## 4) Load and chunk the PDF

LangChain's knowledge-base tutorial describes the load → split → embed → store flow for PDF documents.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))
print("First page preview:")
print(pages[0].page_content[:500])


## 5) Create the ChromaDB index

Chroma is a vector store for embeddings, and LangChain provides a standard Chroma integration.

A quick warning worth internalizing here: if you don't pass Chroma an explicit `embedding_function`, it downloads a default ONNX embedding model the first time it's used — a surprise network call from something that looks like a purely local vector store. We always pass our own embeddings explicitly below, for exactly that reason.

We try the real `sentence-transformers` model first. If it can't be reached — no internet, the package isn't installed, whatever — we fall back to a small dependency-free hashing embedding that exposes the same `embed_documents` / `embed_query` interface. It won't understand synonyms the way a real sentence embedding does, but it's enough to demonstrate the retrieval pipeline mechanically end to end, and Chroma never needs to know which one is active.


In [ ]:
import shutil
import hashlib
import numpy as np
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma


class LocalHashEmbeddings(Embeddings):
    """
    Deterministic, dependency-free fallback embedding function. Used only when
    the real sentence-transformers model can't be reached. It hashes each
    token into a fixed-size bag-of-words vector -- mechanically sound for
    demonstrating similarity search, but semantically much weaker than a real
    sentence embedding (no sense of synonyms or word order).
    """

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str) -> list[float]:
        vec = np.zeros(self.dim)
        for token in text.lower().split():
            h = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = np.linalg.norm(vec)
        return (vec / norm).tolist() if norm > 0 else vec.tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


def build_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        emb.embed_query("connectivity check")  # forces the download/load now, not later
        print("Using live HuggingFace embeddings (sentence-transformers/all-MiniLM-L6-v2).")
        return emb
    except Exception as e:
        print(f"[offline-fallback] HuggingFace embeddings unavailable ({type(e).__name__}: {e}).")
        print("[offline-fallback] Using LocalHashEmbeddings instead.")
        return LocalHashEmbeddings()


embeddings = build_embeddings()

CHROMA_DIR = Path("./data/chroma_ops_store")
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="larkspur_handbook_ops",
    persist_directory=str(CHROMA_DIR),
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma store ready at", CHROMA_DIR.resolve())


## 6) Create the SQLite database

SQLite stores the structured side of the assistant: product metadata, informal notes agents have saved about specific products, and query history.


In [ ]:
import sqlite3
from datetime import datetime, timezone

SQLITE_DB = DATA_DIR / "product_assistant.db"
if SQLITE_DB.exists():
    SQLITE_DB.unlink()

conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

cur.execute("""
CREATE TABLE product_metadata (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    warranty_months INTEGER NOT NULL,
    sku TEXT NOT NULL,
    description TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE product_notes (
    note_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    note_text TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE query_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    intent TEXT NOT NULL,
    source TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

products = [
    ("Aria Wireless Keyboard", "Keyboard", 2499.0, 12, "LK-KB-ARIA-01", "Compact wireless keyboard with quiet switches, pairs over Bluetooth or a USB dongle."),
    ("Comet Wireless Mouse", "Mouse", 899.0, 12, "LK-MS-COMET-01", "Lightweight wireless mouse with a 2000 DPI sensor and a 3-month battery life."),
    ("Halo Over-Ear Headphones", "Headphones", 1499.0, 12, "LK-HP-HALO-01", "Over-ear headphones with active noise cancellation and 30-hour battery life."),
    ("Vista 24-inch Monitor", "Monitor", 3199.0, 24, "LK-MN-VISTA24", "24-inch 1080p IPS monitor with a 75Hz refresh rate."),
    ("Pixel HD Webcam", "Webcam", 1899.0, 12, "LK-WC-PIXEL", "1080p webcam with autofocus and a built-in privacy shutter."),
    ("Glow Desk Lamp", "Desk Lighting", 799.0, 6, "LK-DL-GLOW", "Adjustable LED desk lamp with three brightness settings and USB passthrough."),
]
cur.executemany(
    "INSERT INTO product_metadata (name, category, price, warranty_months, sku, description) VALUES (?, ?, ?, ?, ?, ?)",
    products,
)

notes = [
    ("Aria Wireless Keyboard", "A few customers on Windows 11 report the keyboard dropping Bluetooth pairing after a driver update; re-pairing from Bluetooth settings fixes it."),
    ("Comet Wireless Mouse", "Sensor can feel laggy on glass desks; recommend a mousepad -- it's a surface issue, not a defect."),
    ("Pixel HD Webcam", "Autofocus hunts in low light; suggest customers turn on a desk lamp for calls rather than assuming the unit is faulty."),
]
for product_name, note_text in notes:
    cur.execute(
        "INSERT INTO product_notes (product_name, note_text, created_at) VALUES (?, ?, ?)",
        (product_name, note_text, datetime.now(timezone.utc).isoformat()),
    )

conn.commit()
conn.close()

print("SQLite database created:", SQLITE_DB.resolve())


## 7) Inspect the SQLite data


In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

print("product_metadata rows:")
for row in cur.execute("SELECT name, category, price, warranty_months FROM product_metadata"):
    print(row)

print("\nproduct_notes rows:")
for row in cur.execute("SELECT product_name, substr(note_text, 1, 90) FROM product_notes"):
    print(row)

conn.close()


## 8) Set up Groq

Groq synthesizes the final answer from whichever context the router hands it.


In [ ]:
from langchain_groq import ChatGroq

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

print("Groq model:", GROQ_MODEL)


## 9) Define the graph state

The graph uses a small state object plus a checkpointed thread so the conversation can continue across turns. LangGraph checkpointers save thread state as checkpoints, enabling conversational memory and time travel.


In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    intent: str
    context: str
    source: str
    answer: str


## 10) Routing helpers

Use SQLite for structured questions like price, SKU, category, saved notes, and query history.

Use ChromaDB for policy questions about the handbook's actual content, such as claims, refunds, returns, and shipping delays.


In [ ]:
def latest_user_message(state: AgentState) -> str:
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            return msg.content
        if isinstance(msg, dict) and msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def route_intent(question: str) -> str:
    q = question.lower()
    sqlite_terms = [
        "price", "sku", "spec", "catalog", "category", "note", "saved", "log", "history"
    ]
    policy_terms = [
        "refund", "return", "claim", "replacement", "shipping", "delivery", "delay",
        "escalat", "policy", "handbook", "festival"
    ]

    if any(term in q for term in sqlite_terms):
        return "sqlite"
    if any(term in q for term in policy_terms):
        return "chroma"
    return "chroma"


## 11) Retrieve from SQLite or ChromaDB


In [ ]:
def sqlite_context(question: str) -> str:
    q = question.lower()
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()

    if any(term in q for term in ["price", "sku", "spec", "catalog", "category"]):
        rows = cur.execute("SELECT name, category, price, warranty_months, sku FROM product_metadata").fetchall()
        conn.close()
        return "".join([f"{r[0]} ({r[4]}) | category={r[1]} | price=Rs. {r[2]} | warranty={r[3]} months" for r in rows])

    if "note" in q or "saved" in q:
        rows = cur.execute("SELECT product_name, note_text FROM product_notes ORDER BY note_id DESC LIMIT 5").fetchall()
        conn.close()
        return "".join([f"{r[0]}: {r[1]}" for r in rows])

    if "log" in q or "history" in q:
        rows = cur.execute("SELECT question, intent, source, created_at FROM query_log ORDER BY log_id DESC LIMIT 5").fetchall()
        conn.close()
        return "".join([f"Question: {r[0]} | intent={r[1]} | source={r[2]} | at={r[3]}" for r in rows])

    rows = cur.execute("SELECT name, category, price FROM product_metadata").fetchall()
    conn.close()
    return "".join([f"{r[0]} ({r[1]}) - Rs. {r[2]}" for r in rows])

def chroma_context(question: str) -> str:
    docs = retriever.invoke(question)
    lines = []
    for doc in docs:
        page = doc.metadata.get("page", "n/a")
        preview = doc.page_content[:700].replace("\n", " ")
        lines.append(f"[page={page}] {preview}")
    return "".join(lines)


## 12) Define the answer and log nodes


In [ ]:
def route_node(state: AgentState):
    question = latest_user_message(state)
    intent = route_intent(question)
    return {"intent": intent}

def answer_node(state: AgentState):
    question = latest_user_message(state)
    context = state.get("context", "")
    source = state.get("source", "chroma")

    prompt = (
        "You are CareLoom, a product and policy assistant for Larkspur Electronics.\n\n"
        f"Query source: {source}\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer clearly and concisely. If the context is insufficient, say so."
    )

    try:
        response = llm.invoke(prompt).content.strip()
    except Exception as e:
        response = f"[offline-fallback] LLM unavailable ({type(e).__name__}: {e}). Raw context: {context[:500]}"

    return {
        "answer": response,
        "messages": [AIMessage(content=response)],
    }

def log_node(state: AgentState):
    question = latest_user_message(state)
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO query_log (question, intent, source, created_at) VALUES (?, ?, ?, ?)",
        (question, state.get("intent", ""), state.get("source", ""), datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()
    conn.close()
    return {}


## 13) Build the router graph

The checkpointer stores the thread state so the same thread can be resumed later with the same `thread_id`.


In [ ]:
def route_after_intent(state: AgentState) -> Literal["sqlite_lookup", "chroma_lookup"]:
    return "sqlite_lookup" if state["intent"] == "sqlite" else "chroma_lookup"

def sqlite_lookup_node(state: AgentState):
    question = latest_user_message(state)
    context = sqlite_context(question)
    return {"context": context, "source": "sqlite"}

def chroma_lookup_node(state: AgentState):
    question = latest_user_message(state)
    context = chroma_context(question)
    return {"context": context, "source": "chroma"}

builder = StateGraph(AgentState)

builder.add_node("route", route_node)
builder.add_node("sqlite_lookup", sqlite_lookup_node)
builder.add_node("chroma_lookup", chroma_lookup_node)
builder.add_node("answer", answer_node)
builder.add_node("log", log_node)

builder.add_edge(START, "route")
builder.add_conditional_edges("route", route_after_intent)
builder.add_edge("sqlite_lookup", "answer")
builder.add_edge("chroma_lookup", "answer")
builder.add_edge("answer", "log")
builder.add_edge("log", END)

CHECKPOINT_DB_PATH = DATA_DIR / "agent_checkpoints.db"
if CHECKPOINT_DB_PATH.exists():
    CHECKPOINT_DB_PATH.unlink()

checkpoint_db = sqlite3.connect(str(CHECKPOINT_DB_PATH), check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_db)
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled.")


## 14) Ask a structured SQLite question


In [ ]:
config = {"configurable": {"thread_id": "product-thread-1"}}

result_sqlite = graph.invoke(
    {
        "messages": [HumanMessage(content="What is the price and warranty period of the Aria Wireless Keyboard?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_sqlite


## 15) Ask a semantic policy question


In [ ]:
result_chroma = graph.invoke(
    {
        "messages": [HumanMessage(content="What's the process for claiming a warranty replacement?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_chroma


## 16) Ask about saved notes


In [ ]:
result_notes = graph.invoke(
    {
        "messages": [HumanMessage(content="Show me the latest saved product notes.")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_notes


## 17) Follow-up in the same thread


In [ ]:
follow_up = graph.invoke(
    {
        "messages": [HumanMessage(content="And what about the returns policy for opened items?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

follow_up


## 18) Inspect query history


In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()
rows = cur.execute("SELECT question, intent, source, created_at FROM query_log ORDER BY log_id DESC").fetchall()
conn.close()

rows


## Key takeaways

- SQLite is the structured store for product metadata, saved agent notes, and query history.
- ChromaDB is the semantic store for the policy handbook's actual content.
- A simple router can send a query to the right store based on intent.
- LangGraph checkpointers keep the thread resumable, so the assistant can continue the same conversation later.
- Generating the source PDF locally, instead of downloading one, removes a whole category of flakiness without losing anything the pipeline actually teaches.

### What's verified where

| Piece | Status |
|---|---|
| PDF generation, chunking, SQLite schema and queries, routing logic, checkpointing | Executed and verified in this sandbox — all local, no external dependency. |
| Chroma embeddings | Verified using the `LocalHashEmbeddings` fallback in this sandbox (no HuggingFace network access here). On your machine, with internet access, this will use the real `sentence-transformers` model instead and retrieval quality will be noticeably better. |
| `answer_node`'s Groq call | **Not available in this sandbox** — every run above shows the `[offline-fallback]` raw-context path. Confirm the actual generated answers on your own machine with a real `GROQ_API_KEY`. |

## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Persistence and checkpointers: https://docs.langchain.com/oss/python/langgraph/persistence
- Chroma integration: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- Knowledge base tutorial: https://docs.langchain.com/oss/python/langchain/knowledge-base
